# Amplitude Estimation


Amplitude estimation, in the simplest case, where we have an unknown Bernoulli Random Variable - and we want to estimate its success probability $p$. 

We assume that a qubit is prepared in a state $\sqrt{1-p}\,\big|0\rangle + \sqrt{p}\,\big|1\rangle$, i.e., the probability of measuring $\big|1\rangle$ equals $p$.
<br>
<br>
Amplitude estimation uses two operators: <br>
1) $A$ - describes the problem as, $$A = R_y(\theta_p)$$ i.e., it equals a rotation with an angle $\theta_p = 2\sin^{-1}(\sqrt{p})$ around the Y-axis.<br>
2) $Q$ - is derived from $A$. For this single qubit case, $Q$ is represented by a Y-rotation of twice $\theta_p$, i.e., $$Q = R_y(2\theta_p)$$

This particularly implies that powers of the operator can be efficiently represented, i.e. $Q^k = R_y(2k\theta_p)$.
The result is then mapped into $m$ evaluation qubits. The larger $m$, the higher the accuracy of the estimation but also the longer the quantum circuit.

In [ ]:
# === BACKEND SETUP (qiskit_backend_config) ===
# -----------------------------------------------
# Common backend helper — works with Qiskit 2.x / qiskit-aer / qiskit-ibm-runtime.
# Provides drop-in replacements for the removed Aer, BasicAer, execute(), and IBMQ.
#
# HOW TO SELECT A BACKEND
# -----------------------
# Simulator (default, runs locally — no credentials needed):
#   BACKEND_NAME = "qasm_simulator"        # shot-based measurements
#   BACKEND_NAME = "statevector_simulator" # full statevector
#   BACKEND_NAME = "unitary_simulator"     # full unitary matrix
#
# Real IBM hardware (requires IBM Quantum account):
#   Set the environment variable:  IBM_QUANTUM_TOKEN=<your_token>
#   Then choose a device, e.g.:
#   BACKEND_NAME = "ibm_brisbane"
#   BACKEND_NAME = "least_busy"  # auto-pick least-busy device
#
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from qiskit_backend_config import (get_backend, execute, Aer, BasicAer,
                                   IBMQ, least_busy, job_monitor, assemble, combine)

# ── Choose your backend here ──────────────────────────────────────────────
BACKEND_NAME = "qasm_simulator"  # change to a device name for real hardware
# ── (Optional) IBM Quantum token — leave empty to use env var or saved creds
IBM_QUANTUM_TOKEN = os.environ.get("IBM_QUANTUM_TOKEN", "")

# Build the backend object used throughout this notebook
_token_arg = IBM_QUANTUM_TOKEN if IBM_QUANTUM_TOKEN else None
backend = get_backend(BACKEND_NAME, token=_token_arg)
print(f'Backend ready: {backend}')


In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
from qiskit import QuantumCircuit
from qiskit.visualization import plot_bloch_vector
from qiskit_backend_config import BasicAer
# Aqua's AmplitudeEstimation / CircuitFactory / UncertaintyProblem were removed;
# amplitude estimation now lives in qiskit-algorithms and works on plain circuits + a Sampler
from qiskit_algorithms import AmplitudeEstimation, EstimationProblem
from qiskit.primitives import StatevectorSampler


In [ ]:
# the probability to be recovered
probability = 0.72
theta_p = 2*np.arcsin(np.sqrt(probability))

# the resulting quantum state after A is applied
plot_bloch_vector([np.sin(theta_p), 0.0, np.cos(theta_p)])

In [ ]:
class BernoulliA(QuantumCircuit):
    """
    Circuit representing the operator A.
    A is used to initialize the state as well as to construct Q.
    """

    def __init__(self, probability=0.5):
        super().__init__(1)
        self._theta_p = 2 * np.arcsin(np.sqrt(probability))
        # A is a rotation of angle theta_p around the Y-axis
        self.ry(self._theta_p, 0)


class BernoulliQ(QuantumCircuit):
    """
    Circuit representing the operator Q.
    This implementation exploits the fact that powers of Q can be implemented efficiently by just multiplying the angle.
    (amplitude estimation only requires controlled powers of Q, thus, only power() is overridden.)
    """

    def __init__(self, probability=0.5):
        super().__init__(1)
        self._theta_p = 2 * np.arcsin(np.sqrt(probability))
        # Q is a rotation of angle 2*theta_p around the Y-axis
        self.ry(2 * self._theta_p, 0)

    def power(self, power, *args, **kwargs):
        q_k = QuantumCircuit(1)
        q_k.ry(2 * power * self._theta_p, 0)
        return q_k


In [ ]:
# construct the circuits for A and Q
bernoulli_a = BernoulliA(probability)
print(bernoulli_a)
print(bernoulli_a.num_qubits)
bernoulli_q = BernoulliQ(probability)

# set number of evaluation qubits
# with more ancilla qubits - the estimation is more accurate
#m = 1
m = 5

# construct amplitude estimation
# here, we override the standard construction of Q since we know a more efficient way
# (exploiting the fact that A and Q are just Y-rotations)
problem = EstimationProblem(state_preparation=bernoulli_a, grover_operator=bernoulli_q, objective_qubits=[0])
ae = AmplitudeEstimation(num_eval_qubits=m, sampler=StatevectorSampler())


In [ ]:
result = ae.estimate(problem)


In [ ]:
print(result)

In [ ]:
# plot estimated values
values, probabilities = zip(*sorted(result.samples.items()))
plt.bar(values, probabilities, width=0.5/len(probabilities))
plt.plot([probability, probability], [0,1], 'r--', linewidth=2)
plt.xticks(size=15)
plt.yticks([0, 0.25, 0.5, 0.75, 1], size=15)
plt.title('Estimated Values', size=15)
plt.ylabel('Probability', size=15)
plt.ylim((0,1))
plt.grid()
plt.show()


In [ ]:
# plot circuit
ae.construct_circuit(problem).draw(output='mpl')


## References
<br>
<br>
[1] <a href="https://arxiv.org/abs/1806.06893">Quantum Risk Analysis. Woerner, Egger. 2018.</a>
<br>
[2] <a href="https://arxiv.org/abs/quant-ph/0005055">Quantum Amplitude Amplification and Estimation. Brassard et al. 2000.</a>